In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
EXAMPLE = 'real_robot'
IMG_NAME = 'start pos.png'

# LLAMA_ID = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA_ID = "meta-llama/Meta-Llama-3-70B-Instruct"
# LLAMA_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

# LLAVA_ID = "liuhaotian/llava-v1.6-34b"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-0.5b-si"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-0.5b-ov"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-7b-si"
LLAVA_ID = "lmms-lab/llava-onevision-qwen2-7b-ov"
# LLAVA_ID = "lmms-lab/llava-onevision-qwen2-72b-ov"

import warnings
warnings.filterwarnings("ignore")

In [ ]:
from semantic_state_estimator.semantic_state_estimator import SemanticEstimatorMultiImageRun #SemanticStateEstimatorWithLLaMA

se = SemanticEstimatorMultiImageRun(
    domain=f'examples/{EXAMPLE}/domain.pddl',
    problem=f'examples/{EXAMPLE}/problem.pddl',
    nl_converter_model_id=LLAMA_ID,
    vqa_model_id=LLAVA_ID
)

In [ ]:
from PIL import Image

img = Image.open(f'examples/{EXAMPLE}/{IMG_NAME}')
img

In [ ]:
se([img])

In [ ]:
sorted([w for w in vqa.tokenizer.get_vocab() if 'false' in w.lower()])

In [ ]:
prob_map = se.estimate_state([img])
prob_map

In [ ]:
prob_map = se.estimate_state_par([img], batch_size=16)
prob_map

In [ ]:
state = set()
for pred, prob in prob_map.items():
    if prob > 0.5:
        state.add(pred)

state

In [ ]:
from PIL import ImageDraw, ImageFont

def display_res(query, output, img, save_path=None):
    img = img.copy()
    imd = ImageDraw.Draw(img)
    fnt = ImageFont.truetype("LiberationMono-Regular.ttf", 30)
    imd.text((28, 36), f'{query}: {output}', font=fnt, fill=(255, 0, 255))
    if save_path is None:
        img.show()
    else:
        img.save(save_path)

In [ ]:
from PIL import Image

image_files = [
    'start.png',
    'holding_milk.png',
    'holding_cereal.png',
    'milk_in_container.png',
    'cereal_in_container.png'
]
images = list(map(lambda imgf: Image.open(f'examples/{EXAMPLE}/{imgf}'), image_files))

In [ ]:
from tqdm.notebook import tqdm

for i, image in enumerate(tqdm(images)):
    prob_map = se.estimate_state_par([image], batch_size=4)
    for j, (pred, prob) in enumerate(tqdm(prob_map.items(), leave=False)):
        display_res(pred, f'{prob*100:.2f}%', image, save_path=f'demo/IMG{i}_pred{j}.png')

In [ ]:
for j, (pred, prob) in enumerate(tqdm(prob_map.items(), leave=False)):
    display_res(pred, f'{prob*100:.2f}%', images[-1])

In [ ]:
images[-1]

In [ ]:
img2 = Image.open(f'examples/apt0_body_state/kick-crop.jpeg')

In [ ]:
out = se.vqa_model([[img, img2], [img2, img]], ["describe the images", "describe the images please"])

In [ ]:
img2

In [ ]:
len(prob_map)

In [ ]:
se.queries_dict

In [ ]:
se.vqa_model('examples/apt0_body_state/right-foot-forward.jpeg', "Is the girl's left leg in front of her right leg?")

In [ ]:
probs = se.vqa_model('examples/apt0_body_state/right-foot-forward.jpeg', "Is the girl's left leg in front of her right leg?", get_probs=True)[-1]

In [ ]:
probs = se.vqa_model('examples/apt0_body_state/pre-jump.jpg', "Is the girl's left leg in front of her left leg?", get_probs=True)[-1]

In [ ]:
yes = probs[se.vqa_model.tokenizer.encode(['YES', 'yes', 'Yes'])].sum()
no = probs[se.vqa_model.tokenizer.encode(['NO', 'no', 'No'])].sum()

yes_prob = yes / (yes + no)
yes_prob.item()